# Training an individual-level age predictor

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import torch
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'tutorials':
    REPO_ROOT = REPO_ROOT.parent
DATA_ROOT = Path(os.environ.get('SCDECAGE_DATA_ROOT', REPO_ROOT / 'data' / 'scDecAge_OneK1K')).expanduser().resolve()
PYTHON_BIN = os.environ.get('SCDECAGE_PYTHON', sys.executable)
DEVICE = os.environ.get('SCDECAGE_DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu')
CONFIG = REPO_ROOT / 'configs' / 'config.json'
config = json.loads(CONFIG.read_text())
config.update(config.pop('dataset_settings', {}).get(config['dataset'], {}))

def command_environment():
    env = os.environ.copy()
    candidate_lib = Path(PYTHON_BIN).resolve().parents[1] / 'lib'
    if candidate_lib.is_dir():
        current = env.get('LD_LIBRARY_PATH', '')
        env['LD_LIBRARY_PATH'] = f'{candidate_lib}:{current}' if current else str(candidate_lib)
    return env

## Prepare data

In [ ]:
required = [DATA_ROOT / 'encoder_checkpoint.pth', DATA_ROOT / 'human_gene_vocab.json', DATA_ROOT / 'datasets' / 'OneK1K' / 'donor_splits.csv', DATA_ROOT / 'datasets' / 'OneK1K' / 'cache' / 'gene_ids.int32.npy']
if not all((path.is_file() for path in required)):
    raise FileNotFoundError('Prepare the data with scripts/prepare_dataset.py and set SCDECAGE_DATA_ROOT.')


## Train

In [ ]:
TRAINING_OUTPUT = Path(os.environ.get('SCDECAGE_TRAINING_OUTPUT', REPO_ROOT / 'outputs' / 'OneK1K_training')).expanduser().resolve()
training_command = [PYTHON_BIN, str(REPO_ROOT / 'scripts' / 'train_scdecage.py'), '--config', str(CONFIG), '--data-root', str(DATA_ROOT), '--output-dir', str(TRAINING_OUTPUT), '--device', DEVICE]
completed_files = ['best.pth', 'config.json', 'history.csv', 'metrics.json', 'val_predictions.csv', 'test_predictions.csv']
if all(((TRAINING_OUTPUT / name).is_file() for name in completed_files)):
    saved_config = json.loads((TRAINING_OUTPUT / 'config.json').read_text())
    saved_config.setdefault('pathway_reconstruction_weight', 0.0)
    mismatch = [key for key, value in config.items() if saved_config.get(key) != value]
    if mismatch or saved_config.get('architecture', 'scalar_residual_fusion') != 'scalar_residual_fusion':
        raise ValueError('Saved run has a different configuration; choose a new training output directory.')
    print('Using a completed full OneK1K training run; no retraining in this execution.')
else:
    completed = subprocess.run(training_command, cwd=REPO_ROOT, env=command_environment(), text=True, capture_output=True, check=True)
    print(completed.stdout.strip())
